In [56]:
%pip install ucimlrepo


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [57]:
from ucimlrepo import fetch_ucirepo 
  
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 
  
# data (as pandas dataframes) 
X = bank_marketing.data.features 
y = bank_marketing.data.targets 
  
# metadata 
print(bank_marketing.metadata) 
  
# variable information 
print(bank_marketing.variables) 


{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

In [58]:
import pandas as pd
categorical_cols = ['job', 'marital', 'education', 'contact', 'poutcome']
data=X.copy()
data[categorical_cols] = data[categorical_cols].replace('unknown', pd.NA)
print(data.isna().sum())

age                0
job              288
marital            0
education       1857
default            0
balance            0
housing            0
loan               0
contact        13020
day_of_week        0
month              0
duration           0
campaign           0
pdays              0
previous           0
poutcome       36959
dtype: int64


In [59]:
import numpy as np
binary_map = {"yes": 1, "no": 0}
bin_cols = [c for c in ["default", "housing", "loan", "y"] if c in data.columns]
for c in bin_cols:
    data[c] = data[c].map(binary_map).astype("Int64")

In [60]:
cat_cols_raw = [c for c in ["job","marital","education","contact","month","day_of_week","poutcome"] if c in data.columns]
data[cat_cols_raw] = data[cat_cols_raw].replace({"unknown": pd.NA, "UNK": pd.NA, "": pd.NA})

if "pdays" in data.columns:
    data["pdays_missing"] = (data["pdays"] == -1).astype(int)
    data.loc[data["pdays"] == -1, "pdays"] = np.nan

for c in cat_cols_raw:
    if data[c].isna().any():
        data[c] = data[c].fillna(data[c].mode(dropna=True)[0])

num_cols_raw = [c for c in data.select_dtypes(include=["int64","Int64","float64"]).columns if c not in []]
for c in num_cols_raw:
    if data[c].isna().all():
        data[c] = data[c].fillna(0)
    if data[c].isna().any():
        data[c] = data[c].fillna(data[c].median())

In [61]:
if "month" in data.columns:
    month_order = ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]
    data["month"] = pd.Categorical(data["month"], categories=month_order, ordered=True)

if "day_of_week" in data.columns:
    dow_order = ["mon","tue","wed","thu","fri"]
    data["day_of_week"] = pd.Categorical(data["day_of_week"], categories=dow_order, ordered=True)

cat_cols = list(data.select_dtypes(include=["object","category"]).columns)
data = pd.get_dummies(data, columns=cat_cols, drop_first=True, dtype=float)
print(data.shape)

(45211, 42)


In [62]:
from sklearn.preprocessing import StandardScaler
numeric_cols = list(data.select_dtypes(include=["int64","Int64","float64"]).columns)
cols_to_scale = [c for c in numeric_cols if data[c].dropna().nunique() > 2]

scaler = StandardScaler()
data[cols_to_scale] = scaler.fit_transform(data[cols_to_scale])

print("Scaled columns:", cols_to_scale)
data.head()


Scaled columns: ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']


,age,default,balance,housing,loan,duration,campaign,pdays,previous,pdays_missing,...,month_may,month_jun,month_jul,month_aug,month_sep,month_oct,month_nov,month_dec,poutcome_other,poutcome_success
0,1.606965,0,0.256419,1,0,0.011016,-0.569351,-0.110178,-0.25194,1,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.288529,0,-0.437895,1,0,-0.416127,-0.569351,-0.110178,-0.25194,1,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.747384,0,-0.446762,1,1,-0.707361,-0.569351,-0.110178,-0.25194,1,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.571051,0,0.047205,1,0,-0.645231,-0.569351,-0.110178,-0.25194,1,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.747384,0,-0.447091,0,0,-0.233620,-0.569351,-0.110178,-0.25194,1,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [63]:
y=y.squeeze()
y=y.astype(str).str.strip().str.lower()
y=y.map(binary_map).astype("Int64")
print(y)

0        0
1        0
2        0
3        0
4        0
        ..
45206    1
45207    1
45208    1
45209    0
45210    0
Name: y, Length: 45211, dtype: Int64


In [64]:
X=np.array(data)
y=np.array(y)

In [ ]:
import matplotlib.pyplot as plt

class svm:
    def __init__(self, visualization=True):
        self.visualization = visualization
        self.colors = {1: 'r', -1: 'b'}
        if self.visualization:
            self.fig=plt.figure()
            self.ax = self.fig.add_subplot(1, 1, 1)
    
    def fit(self, X, y):
        pass
    
    def predict(self, X):
        classification = np.sign(np.dot(X, self.w) + self.b)
        return classification
        